In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

def fetch(start_date="2010-01-01"):
    
    tickers = ["SPY", "EFA", "TLT", "GLD", "BIL"]
    raw_data = yf.download(tickers, start=start_date)['Close']

    data = raw_data.ffill().dropna(how='all')
    return data

def multi_signals(prices_df):
    
    ret_1 = prices_df.pct_change(21)
    ret_3 = prices_df.pct_change(63)
    ret_6 = prices_df.pct_change(126)
    ret_12 = prices_df.pct_change(252)

    blended_momentum = (ret_1 + ret_3 + ret_6 + ret_12) / 4.0

    eom_blend = blended_momentum.resample('BME').last()

    risk_assets = ['SPY', 'EFA', 'TLT', 'GLD']
    safe_asset = 'BIL'

    ranks = eom_blend[risk_assets].rank(axis=1, ascending=False)
    top_asset = (ranks == 1.0)

    positive = (eom_blend[risk_assets] > 0)

    buy_signals = top_asset & positive

    target = buy_signals.astype(float)

    target[safe_asset] = 1.0 - target.sum(axis=1)

    return target.dropna()

In [2]:
def regime_overlay(prices_df, target):

    spy_close = prices_df['SPY']
    spy_200 = spy_close.rolling(window=200).mean()

    risk_off = spy_close < spy_200

    eom_risk_off = risk_off.resample('BME').last()

    regime_adjusted = target.copy()

    risk_assets = ['SPY', 'EFA', 'TLT', 'GLD']
    safe_asset = 'BIL'

    eom_risk_off = eom_risk_off.reindex(regime_adjusted.index).fillna(False)

    regime_adjusted.loc[eom_risk_off, risk_assets] = 0.0

    regime_adjusted.loc[eom_risk_off, safe_asset] = 1.0

    months_triggered = eom_risk_off.sum()
    print(f"Circuit Breaker triggered in {months_triggered} months")

    return regime_adjusted

In [3]:
def vectorized_backtest(prices_df, target):
    
    prices = prices_df.resample('BME').last()
    returns = prices.pct_change()

    shifted = target.shift(1)

    portfolio_returns = (shifted * returns).sum(axis=1)

    benchmark_returns = returns['SPY']

    portfolio_equity = (1 + portfolio_returns).dropna().cumprod()
    benchmark_equity = (1 + benchmark_returns.fillna(0)).cumprod()

    equity_curves = pd.DataFrame({
        'Systematic_Macro_Fund': portfolio_equity,
        'Benchmark_SPY': benchmark_equity
    })

    return equity_curves, portfolio_returns.dropna(), benchmark_returns.dropna()

def kpis(returns_series, equity_series, name="Strategy"):
    
    months = len(returns_series)
    years = months / 12
    total_return = equity_series.iloc[-1]
    cagr = (total_return ** (1 / years)) - 1

    mean_ret = returns_series.mean()
    std_ret = returns_series.std()
    sharpe = (mean_ret / std_ret) * np.sqrt(12)

    rolling_max = equity_series.cummax()
    drawdowns = (equity_series - rolling_max) / rolling_max
    max_dd = drawdowns.min()

    print(f"\n{name} Performance")
    print(f"Annualized Return (CAGR): {cagr:.2%}")
    print(f"Annualized Sharpe Ratio:  {sharpe:.2f}")
    print(f"Maximum Drawdown:         {max_dd:.2%}")

    return cagr, sharpe, max_dd

In [4]:
def volatility(prices_df, base, target_vol=0.15, max_leverage=1.0):

    daily_returns = prices_df.pct_change()

    rolling_vol = daily_returns.rolling(window=20).std() * np.sqrt(252)

    eom_vol = rolling_vol.resample('BME').last()

    eom_vol = eom_vol.reindex(base.index).ffill()

    position_sizes = target_vol / eom_vol

    position_sizes = position_sizes.clip(upper=max_leverage)

    vol_adjusted = base * position_sizes

    risk_assets = ['SPY', 'EFA', 'TLT', 'GLD']
    invested_capital = vol_adjusted[risk_assets].sum(axis=1)

    if 'BIL' not in vol_adjusted.columns:
        vol_adjusted['BIL'] = 0.0

    vol_adjusted['BIL'] = vol_adjusted['BIL'] + (1.0 - invested_capital)

    return vol_adjusted

In [5]:
def correlation(prices_df, target, lookback=60, danger_threshold=0.4):

    daily_returns = prices_df.pct_change()

    rolling_corr = daily_returns.rolling(window=lookback).corr()

    def avg_corr(matrix):
        upper_tri = matrix.values[np.triu_indices_from(matrix.values, k=1)]
        return np.nanmean(upper_tri)

    daily_avg_corr = rolling_corr.groupby(level=0).apply(avg_corr)

    eom_avg_corr = daily_avg_corr.resample('BME').last()
    eom_avg_corr = eom_avg_corr.reindex(target.index).ffill()

    multiplier = np.where(eom_avg_corr > danger_threshold, 0.5, 1.0)
    multiplier_series = pd.Series(multiplier, index=target.index)

    adjusted = target.copy()
    risk_assets = ['SPY', 'EFA', 'TLT', 'GLD']

    for asset in risk_assets:
        if asset in adjusted.columns:
            adjusted[asset] = adjusted[asset] * multiplier_series

    invested_capital = adjusted[risk_assets].sum(axis=1)
    adjusted['BIL'] = 1.0 - invested_capital

    spikes_detected = (multiplier_series == 0.5).sum()
    print(f"Systemic risk averted in {spikes_detected} months")

    return adjusted

In [6]:
def monte_carlo(prices_df, target, num_sims=10000, horizon_days=252):

    current = target.iloc[-1]
    assets = current.index

    print(f"Current Target Allocation:\n{current[current > 0].to_string()}")

    daily_returns = prices_df[assets].pct_change().dropna()

    mu_horizon = daily_returns.mean().values * horizon_days
    cov_horizon = daily_returns.cov().values * horizon_days

    L = np.linalg.cholesky(cov_horizon)

    Z = np.random.normal(0, 1, size=(num_sims, len(assets)))

    simulated_asset = mu_horizon + Z.dot(L.T)

    simulated_portfolio = simulated_asset.dot(current.values)

    var_99 = np.percentile(simulated_portfolio, 1)

    cvar_99 = simulated_portfolio[simulated_portfolio <= var_99].mean()

    print(f"99% Value at Risk (VaR):          {var_99:.2%}")
    print(f"99% Expected Shortfall (CVaR):    {cvar_99:.2%}")

    return simulated_portfolio, var_99, cvar_99

In [7]:
def risk_of_ruin(returns_series, ruin_threshold=-0.50, num_sims=10000):
    
    print(f"Catastrophic Failure Threshold defined as: {ruin_threshold:.0%}")

    historical_returns = returns_series.values
    months = len(historical_returns)

    simulated_paths = np.random.choice(historical_returns, size=(num_sims, months), replace=True)

    equity_curves = np.cumprod(1 + simulated_paths, axis=1)

    running_max = np.maximum.accumulate(equity_curves, axis=1)

    drawdowns = (equity_curves - running_max) / running_max

    max_drawdowns = np.min(drawdowns, axis=1)

    ruined_paths = np.sum(max_drawdowns <= ruin_threshold)

    prob_of_ruin = ruined_paths / num_sims

    print(f"Paths ending in Ruin: {ruined_paths:,} out of {num_sims:,}")
    print(f"Probability of Ruin:  {prob_of_ruin:.2%}")

    if prob_of_ruin > 0.05:
        print("Risk of Ruin exceeds 5%. The system is over-leveraged")
    else:
        print("The architecture is highly robust to sequence risk")

    return prob_of_ruin, max_drawdowns

In [8]:
if __name__ == "__main__":
    macro_df = fetch(start_date="2007-01-01")

    base = multi_signals(macro_df)

    vol = volatility(macro_df, base, target_vol=0.15)

    corr = correlation(macro_df, vol, danger_threshold=0.4)

    final = regime_overlay(macro_df, corr)

    equity_df, port_ret, bench_ret = vectorized_backtest(macro_df, final)

    kpis(port_ret, equity_df['Systematic_Macro_Fund'], name="Systematic Macro Fund")
    kpis(bench_ret, equity_df['Benchmark_SPY'], name="Benchmark (S&P 500)")

    monte_carlo(macro_df, final)

    risk_of_ruin(port_ret, ruin_threshold=-0.50)

[*********************100%***********************]  5 of 5 completed
C:\Users\HP\AppData\Local\Temp\ipykernel_3848\2678830822.py:9: RuntimeWarning: Mean of empty slice
  return np.nanmean(upper_tri)


Systemic risk averted in 1 months
Circuit Breaker triggered in 49 months

Systematic Macro Fund Performance
Annualized Return (CAGR): 9.13%
Annualized Sharpe Ratio:  0.81
Maximum Drawdown:         -21.35%

Benchmark (S&P 500) Performance
Annualized Return (CAGR): 10.38%
Annualized Sharpe Ratio:  0.72
Maximum Drawdown:         -50.78%
Current Target Allocation:
Ticker
BIL    0.437253
GLD    0.562747
99% Value at Risk (VaR):          -15.58%
99% Expected Shortfall (CVaR):    -18.69%
Catastrophic Failure Threshold defined as: -50%
Paths ending in Ruin: 13 out of 10,000
Probability of Ruin:  0.13%
The architecture is highly robust to sequence risk


In [9]:
import quantstats as qs
from IPython.display import HTML

qs.extend_pandas()
qs.reports.html(port_ret, benchmark="SPY", title="Systematic Macro Strategy", output='report.html')

qs.reports.metrics(port_ret, benchmark="SPY")

with open('report.html', 'r') as f:
    html_content = f.read()
HTML(html_content)


Parameter       Value
--------------  ---------------
Benchmark       Benchmark (SPY)
Risk-Free Rate  0.0%
Periods/Year    252
Compounded      Yes
Match Dates     Yes


                     Benchmark (SPY)    Strategy
-------------------  -----------------  ----------
Start Period         2007-06-29         2007-06-29
End Period           2026-03-31         2026-03-31
Risk-Free Rate       0.0%               0.0%
Time in Market       100.0%             99.0%

Cumulative Return    533.86%            437.29%
CAGR﹪               683.89%            551.96%

Sharpe               3.27               3.77
Prob. Sharpe Ratio   99.82%             99.99%
Sortino              4.92               6.9
Sortino/√2           3.48               4.88
Omega                1.69               1.98

Max Drawdown         -50.78%            -21.35%
Max DD Date          2009-02-27         2019-05-31
Max DD Period Start  2007-11-30         2018-10-31
Max DD Period End    2012-02-29         2021-03-31
Longest DD D